In [6]:
#!pip install sympy

In [7]:
from pyomo.environ import *
import numpy as np
from math import pi
from Sympy_OPF_LALM_class import SympyACOPFModel
from Sympy_OPF_LALM_class import PrintQHDACOPFResults
from Sympy_OPF_LALM_class import solve_with_gurobi_from_sympy
from Sympy_OPF_LALM_class import extract_qhd_solution_vector
from Sympy_OPF_LALM_class import initialize_qhd_acopf_log
from Sympy_OPF_LALM_class import append_qhd_acopf_results
from qhdopt import QHD
import json

In [8]:
def load_matpower_json(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    Sbase = float(data["Sbase"])

    # Convert "k1", "k2", ... into integer keys 1, 2, ...
    buses = {int(k.replace("k", "")): v for k, v in data["buses"].items()}
    lines = {int(k.replace("k", "")): v for k, v in data["lines"].items()}
    gens  = {int(k.replace("k", "")): v for k, v in data["gens"].items()}

    return Sbase, buses, lines, gens


In [9]:
# Initialize the model.
# model = SympyACOPFModel()   # Enable the proximal option if needed later.
n = 3 # Select the test system size: 2, 3, or larger.

if n == 2:
    # 2-bus model
    Sbase = 10.0
    buses = {
        1: [1, 0, 1.00, 0.0, 0.0, 0.0, 0.0, 0.0],
        2: [2, 1, 1.01, 0.0, 0.0, 0.0, 0.3, 0.1],
    }
    lines = {
        1: [1, 2, 0.0452, 0.1852, 0.0204, 1.0, 30.0 / Sbase],
    }
    gens = {
        1: [1, 0.0 / Sbase, 20.0 / Sbase, -20.0 / Sbase, 100.0 / Sbase, 0.00375, 2.0, 0.0],
    }
    model = SympyACOPFModel(Sbase=Sbase, buses=buses, lines=lines, gens=gens)

elif n == 3:
    # 3-bus model
    model = SympyACOPFModel()

else:
    # n-bus model
    Sbase, buses, lines, gens = load_matpower_json(f"case{n}_custom_pretty.json")
    model = SympyACOPFModel(Sbase=Sbase, buses=buses, lines=lines, gens=gens)

print("Model initialized with", n, "buses", model.n_lines, "lines and", model.n_gens, "generators.")


Model initialized with 3 buses 3 lines and 2 generators.


In [ ]:
# =========================
#   Linear ALM + QHD Loop
# =========================

import numpy as np

# 1) 构造 h(x)
h_func = model.build_h_func()
model.reset_lambdas(0.0)

# 2) 初始点
xk = model.build_initial_x0()
#xk = np.load("opf_result_ordered.npy", allow_pickle=True)

rho = 5.0
alpha = 1e-4   # 对偶步长尽量小一点
max_outer = 100
tol = 1e-4
option = 1  # 1: QHD, 2: Gurobi
qhd_solver = "simbi"  # openjij / simbi

# ========= 新增：初始化日志文件 =========
log_file = initialize_qhd_acopf_log(model, folder="logs", option=option, qhd_solver=qhd_solver)
print("Log file:", log_file)

print("\n===== Start Linear ALM Loop =====\n")

for k in range(max_outer):

    print(f"\n--- Outer Iteration {k} ---")

    # ======================================
    # (1) 在线性化点 xk 构造二次 L^(k)(x)
    # ======================================
    Lag, variable_list, Var_bound_list = \
        model.build_linear_ALM_Lagrangian_syms(
            x_center=xk,
            rho=rho,
            ref_bus_id=None,
            mu_prox=20.0
        )
    
    bad_bounds = []
    for i, (var, bnd) in enumerate(zip(variable_list, Var_bound_list)):
        lb, ub = float(bnd[0]), float(bnd[1])
        if ub < lb:
            bad_bounds.append((i, str(var), lb, ub))

    if bad_bounds:
        print("\n=== Invalid bounds found ===")
        for item in bad_bounds:
            print(item)
        raise ValueError("Var_bound_list contains invalid bounds (ub < lb).")

    if option == 1:
        # ======================================
        # (2A) QHD 求解子问题
        # ======================================
        qhd_model = QHD.SymPy(Lag, variable_list, Var_bound_list)


        if qhd_solver == "simbi":
            qhd_model.simbi_setup(
                resolution=7,
                agents=1024,
                max_steps=5000,
                embedding_scheme="unary",
                post_processing_method='TNC',
                penalty_coefficient=0.1,
                ballistic=True,
                heated=True,
                best_only=False,
                verbose=True
            )
        else:
            qhd_model.openjij_setup(
                resolution=6,
                shots=2048,
                sampler_name="SQASampler",
                seed=42,
                debug=False,
                sampler_init_kwargs={},
                sample_kwargs={
                    "beta": 5.0,
                    "gamma": 1.0,
                    "trotter": 8,
                    "num_sweeps": 10000,
                    "reinitialize_state": True,
                },
            )
        response = qhd_model.optimize(refine=True, verbose=0)
        #response = qhd_model.optimize(refine=False, verbose=0)

        x_new = extract_qhd_solution_vector(response, expected_len=len(variable_list))

    else:
        # ======================================
        # (2B) 用 Gurobi 求解子问题
        # ======================================
        x_new = solve_with_gurobi_from_sympy(
        L_sym=Lag,
        variable_list=variable_list,
        Var_bound_list=Var_bound_list,
        verbose=False   # 如果想看 Gurobi 日志就改成 True
    )
    # ======================================
    # (3) 计算真实约束
    # ======================================
    h_val = h_func(x_new)
    norm_h = np.linalg.norm(h_val)

    print("||h(x)|| =", norm_h)

    # ======================================
    # (4) 对偶更新
    # ======================================
    lambda_new, h_x = model.update_lambda(
        x_new,
        alpha=rho,
        h_func=h_func
    )
    # ======================================
    # (5) 自适应 rho
    # ======================================
    h_old = h_func(xk)
    print(f"[rho-check] ||h_old||={np.linalg.norm(h_old):.3e}, ||h_new||={np.linalg.norm(h_val):.3e}, rho={rho:.3g}")
    rho_max = 1024
    if np.linalg.norm(h_x) > 0.5 * np.linalg.norm(h_old) and rho < rho_max:
        rho *= 2
        print("Increasing rho to", rho)

    # ======================================
    # (6) 可行性检查
    # ======================================
    _, check_flag = model.check_constraints(xk)
    print("Constraint check:", check_flag)
    if check_flag:
        print("\nConverged!")
        break

    # ======================================
    # (7) 记录日志（每轮追加）
    # ======================================
    subs_dict = {var: val for var, val in zip(model.variable_list, x_new)}
    #objective_value = float(model.objective.evalf(subs=subs_dict))

    log_file = PrintQHDACOPFResults(
        model,
        x_new,
        log_file=log_file,
        iteration=k,
        folder="logs",
        print_to_console=True,
        rho=rho,
        alpha=alpha,
        h_x=h_val,
        lambda_vec=lambda_new,
        objective_value=0,
        feasibility=check_flag,
    )

    # 如果你还想同时在屏幕上打印表格，可以再保留这一句
    # PrintQHDACOPFResults(model, x_new, iteration=k, log_file=log_file, print_to_screen=True)


    # ======================================
    # (8) 收敛判据
    # ======================================
    step_norm = np.linalg.norm(x_new - xk)
    if norm_h < tol and step_norm < 1e-5:
        print("\nConverged!")
        break

    # ======================================
    # (9) 更新 primal
    # ======================================
    xk = x_new.copy()

print("\n===== End Loop =====\n")
print("Final log file:", log_file)

Log file: logs\Buses-3_04-13-2026_20-00-36.txt

===== Start Linear ALM Loop =====


--- Outer Iteration 0 ---
Set parameter Username
Set parameter LicenseID to value 2752566
||h(x)|| = 0.18837980131935297
[rho-check] ||h_old||=5.181e-01, ||h_new||=1.884e-01, rho=5
Constraint check: False
Update time: 2026-04-13 20:00:39
Iteration: 0
rho: 5
alpha: 0.005
objective_value: 0
feasible: False
max_abs_h: 1.153513511432e-01
l2_norm_h: 1.883798013194e-01
lambda_inf_norm: 5.767567557158e-01
lambda_l2_norm: 9.418990065968e-01
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	1.005488	0.004297	1.001828	0.102529	0.027933	0.000000	0.000000
2	1.005586	0.004227	1.008528	0.105720	0.028369	0.000000	0.000000
3	0.995135	-0.010716	0.998377	0.000000	0.000000	0.300000	0.100000

Branch Results
LineID	From	To	Pik		Pki		Qik		Qki		LossP		LossQ
1	1	2	0.000325	-0.000495	-0.009889	-0.008417	-0.0

KeyboardInterrupt: 